# Naming sanitization
Notebook to sanitize all names for packages, rasters, etc.

## Modules

In [11]:
import pandas as pd
import geopandas as gpd
import fiona
import pyogrio

In [1]:
geom_layer = "geometry_layer"

In [41]:
rename_cols = {
    'cf': 'leaf', 
    'cf_median': 'leaf_median', 
    'cf_std': 'leaf_std', 
}

rename_metrics = {
    'cf_mean': 'leaf_mean', 
    'cf_median': 'leaf_median', 
    'cf_std': 'leaf_std', 
}

In [12]:
def inspect_gpkg(filepath: str):
    layer_info = pyogrio.list_layers(filepath)  # array of [name, geom_type]

    for name, geom_type in layer_info:
        if geom_type is None:
            # non-spatial table -> read without geometry
            df = gpd.read_file(filepath, layer=name, ignore_geometry=True)
            print(f"{name}: TABLE, shape={df.shape}, columns={list(df.columns)}")
        else:
            gdf = gpd.read_file(filepath, layer=name)
            print(f"{name}: {gdf.shape}, CRS={gdf.crs}, geom={gdf.geom_type.unique()}, geom_type_reported={geom_type}")

In [25]:
def load_gpkg(filepath: str):
    layer_info = pyogrio.list_layers(filepath)
    
    data = {}
    
    for name, geom_type in layer_info:
        if geom_type is None:
            df = gpd.read_file(filepath, layer=name, ignore_geometry=True)
        else:
            df = gpd.read_file(filepath, layer=name)
        data[name] = df
    
    return data

In [31]:
def update_gpkg_to_disk(data: gpd.GeoDataFrame, gpkg_path: str):
    first = True
    for name, df in data.items():
        mode = "w" if first else "a"
        if isinstance(df, gpd.GeoDataFrame):
            df.to_file(gpkg_path, layer=name, driver="GPKG", mode=mode)
        else:
            # non-spatial table: needs pyogrio directly since to_file expects geometry
            pyogrio.write_dataframe(df, gpkg_path, layer=name, driver="GPKG", append=(mode == "a"))
        first = False

    print(f'Geopackage updated into: {gpkg_path}')

## Acidification

### Ecoregion

In [35]:
acid_er_fp = "../LEAFs/acidification/acidification_ecoregion.gpkg"
acid_er_df_fp = "../LEAFs/acidification/acidification_subcountry.csv"

In [28]:
inspect_gpkg(acid_er_fp)

geometry_layer: (829, 8), CRS=EPSG:6933, geom=['MultiPolygon'], geom_type_reported=MultiPolygon
acid_leaf_ecoregions: TABLE, shape=(2487, 6), columns=['ECO_ID', 'flow_name', 'cf', 'cf_median', 'cf_std', '_source_file']
acid_leaf_ecoregions_metadata: TABLE, shape=(3, 4), columns=['flow_name', 'impact_category', 'unit', 'source_file']


In [29]:
ac_er_gpkg_data = load_gpkg(acid_er_fp)

In [30]:
ac_er_gpkg_data['acid_leaf_ecoregions'] = ac_er_gpkg_data['acid_leaf_ecoregions'].rename(columns = rename_cols)

In [32]:
update_gpkg_to_disk(ac_er_gpkg_data, acid_er_fp)

Geopackage updated into: ../LEAFs/acidification/acidification_ecoregion.gpkg


#### df

In [36]:
acid_er_df = pd.read_csv(acid_er_df_fp)

In [42]:
acid_er_df["metric"] = acid_er_df["metric"].replace(rename_metrics)

In [43]:
acid_er_df["metric"].unique()

array(['leaf_mean', 'leaf_median', 'leaf_std'], dtype=object)

In [46]:
acid_er_df

,ADM0_NAME,ADM1_NAME,ADM1_CODE,flow_name,metric,value
0,Burundi,Bubanza,40542,acid_nh3,leaf_mean,0.818360
1,Burundi,Bujumbura Mairie,40543,acid_nh3,leaf_mean,0.794068
2,Burundi,Bujumbura Rural,40544,acid_nh3,leaf_mean,0.794068
3,Burundi,Bururi,40545,acid_nh3,leaf_mean,0.794068
4,Burundi,Cankuzo,40546,acid_nh3,leaf_mean,1.038711
...,...,...,...,...,...,...
30793,French Polynesia,Administrative unit not available,1273,acid_nox,leaf_std,0.001042
30794,Pitcairn,Administrative unit not available,2369,acid_nox,leaf_std,0.000714
30795,Niue,Administrative unit not available,2241,acid_nox,leaf_std,0.000206
30796,Cook Islands,Administrative unit not available,980,acid_nox,leaf_std,0.000956


In [47]:
acid_er_df.to_csv(acid_er_df_fp, index=False)

In [48]:
pd.read_csv(acid_er_df_fp).head()

,ADM0_NAME,ADM1_NAME,ADM1_CODE,flow_name,metric,value
0,Burundi,Bubanza,40542,acid_nh3,leaf_mean,0.818360
1,Burundi,Bujumbura Mairie,40543,acid_nh3,leaf_mean,0.794068
2,Burundi,Bujumbura Rural,40544,acid_nh3,leaf_mean,0.794068
3,Burundi,Bururi,40545,acid_nh3,leaf_mean,0.794068
4,Burundi,Cankuzo,40546,acid_nh3,leaf_mean,1.038711
